In [0]:
%sql
-- Databricks notebook source
-- MAGIC %md
-- MAGIC # Gold — dimension tables
-- MAGIC `dim_event`, `dim_athlete`, `dim_date` from silver OBT.

-- COMMAND ----------

CREATE OR REPLACE TABLE marathos.gold.dim_event AS
SELECT
  event_id,
  event_name,
  event_year,
  ROUND(AVG(distance_km), 2) AS distance_km,
  MAX(distance_unit) AS distance_unit,
  CASE
    WHEN MAX(distance_unit) = 'mi' OR LOWER(MAX(event_name)) LIKE '%marathon%' AND AVG(distance_km) < 35 THEN 'marathon_mi'
    WHEN AVG(distance_km) BETWEEN 41 AND 44 THEN 'marathon_km'
    ELSE 'other'
  END AS event_type
FROM marathos.silver.races_obt
GROUP BY event_id, event_name, event_year;

-- COMMAND ----------

CREATE OR REPLACE TABLE marathos.gold.dim_athlete AS
SELECT
  athlete_id,
  MAX(athlete_country) AS athlete_country,
  MAX(athlete_gender) AS athlete_gender,
  MAX(athlete_birth_year) AS athlete_birth_year,
  MAX(athlete_age_category) AS athlete_age_category
FROM marathos.silver.races_obt
GROUP BY athlete_id;

-- COMMAND ----------

CREATE OR REPLACE TABLE marathos.gold.dim_date AS
SELECT
  DENSE_RANK() OVER (ORDER BY event_date, event_year) AS date_id,
  event_date,
  event_year,
  MONTH(event_date) AS event_month,
  DATE_FORMAT(event_date, 'EEEE') AS weekday
FROM (
  SELECT DISTINCT event_date, event_year
  FROM marathos.silver.races_obt
  WHERE event_date IS NOT NULL
);

-- COMMAND ----------

SELECT 'dim_event' AS t, COUNT(*) AS n FROM marathos.gold.dim_event
UNION ALL
SELECT 'dim_athlete', COUNT(*) FROM marathos.gold.dim_athlete
UNION ALL
SELECT 'dim_date', COUNT(*) FROM marathos.gold.dim_date;
